# Riconoscere le varietà di riso con una rete convoluzionale (CNN)



> ⚙️ **Prima di iniziare:** in Colab andate su *Runtime → Cambia tipo di runtime → GPU (T4)*. Con le immagini la GPU fa una grande differenza.

---

## 0. Il problema

Un'azienda che confeziona riso riceve chicchi di varietà diverse e vuole **riconoscere automaticamente la varietà** da una foto del chicco, per controllare la qualità ed evitare miscele.

**Obiettivo:** data la foto di un singolo chicco, dire a quale delle **5 varietà** appartiene.

In termini di machine learning:
- **Apprendimento supervisionato**: abbiamo foto già etichettate.
- **Classificazione multi-classe**: 5 classi possibili.
- L'input non è più una riga di tabella ma un'**immagine**.
## Il dataset

Usiamo il **Rice Image Dataset** (Koklu et al., 2021), molto noto anche su Kaggle.

| | |
|---|---|
| Immagini totali | **75.000** |
| Classi | **5**: Arborio, Basmati, Ipsala, Jasmine, Karacadag |
| Immagini per classe | 15.000 (dataset **bilanciato**, al contrario del churn) |
| Formato | JPG a colori, 250 × 250 pixel, un chicco su sfondo nero |

> Per non perdere troppo tempo useremo **solo una parte** delle immagini (2.000 per classe). Le immagini che restano ci serviranno alla fine come "chicchi nuovi" da riconoscere.

## Cosa cambia rispetto al notebook del churn

| | Churn (notebook 1) | Riso (oggi) |
|---|---|---|
| Input | 30 numeri per cliente | 64 × 64 × 3 = **12.288** numeri per immagine |
| Output | 1 probabilità (sì/no) | **5** probabilità, una per varietà |
| Modello | regressione logistica, MLP | **rete convoluzionale (CNN)** |
| Loss | `BCEWithLogitsLoss` | `CrossEntropyLoss` |
| Addestramento | full-batch | **mini-batch** con `DataLoader` |
| Hardware | CPU | **GPU** |

## Il piano della lezione

1. Importare le librerie
2. Scaricare ed esplorare le immagini
3. Preparare i dati: da immagine a tensore, split, normalizzazione, `DataLoader`
4. Cos'è una **convoluzione** (e il pooling)
5. Costruire la **CNN**
6. Loss per più classi: **softmax** e **cross-entropy**
7. Addestramento con **mini-batch** e early stopping
8. Valutazione sul test set
9. Cosa "vede" la rete
10. Salvare e caricare il modello
11. **Inferenza** su chicchi mai visti
12. **Esercizio**: costruisci la tua CNN
---

## 1. Importare le librerie

| Libreria | A cosa serve qui |
|---|---|
| `torch`, `torch.nn` | Tensori, layer, loss, ottimizzatori |
| `torch.nn.functional` (`F`) | Le stesse operazioni di `nn` ma come funzioni: ci serve per provare una convoluzione "a mano" |
| `torch.utils.data` | `TensorDataset` e `DataLoader`: dividono i dati in mini-batch |
| `PIL` (Pillow) | Aprire, ridimensionare e ruotare immagini |
| `zipfile`, `urllib`, `io` | Scaricare il file zip e leggere le immagini **direttamente dallo zip**, senza estrarlo |
| `numpy` | Passaggio intermedio tra immagine e tensore |
| `matplotlib` | Grafici e visualizzazione delle immagini |
| `sklearn.metrics` | Matrice di confusione e report finale |

In [ ]:
import os, io, time, copy, random, zipfile, urllib.request

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)
if device == "cpu":
    print("⚠️ Nessuna GPU: Runtime -> Cambia tipo di runtime -> GPU. Funziona anche su CPU, ma più lentamente.")

## 2. Scaricare ed esplorare le immagini

Il file zip (~230 MB) viene scaricato **dentro Colab**, non sul vostro computer. Ci vuole circa un minuto.

In [ ]:
URL = "https://huggingface.co/datasets/miladfa7/Rice-Image-Dataset/resolve/main/Rice%20Image%20Dataset.zip"
ZIP_PATH = "rice.zip"

if not os.path.exists(ZIP_PATH):            # se lo abbiamo già scaricato, non lo riscarichiamo
    t0 = time.time()
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print(f"Scaricato in {time.time() - t0:.0f} s")

print(f"Dimensione: {os.path.getsize(ZIP_PATH) / 1e6:.0f} MB")

In [ ]:
archivio = zipfile.ZipFile(ZIP_PATH)

# Tutti i file immagine nello zip. La cartella che li contiene è il nome della classe:
# es. "Rice_Image_Dataset/Arborio/Arborio (1).jpg" -> classe "Arborio"
file_immagini = [n for n in archivio.namelist()
                 if n.lower().endswith((".jpg", ".jpeg", ".png")) and "__MACOSX" not in n]

per_classe = {}
for nome in file_immagini:
    classe = nome.split("/")[-2]
    per_classe.setdefault(classe, []).append(nome)

CLASSI = sorted(per_classe)
for c in CLASSI:
    print(f"{c:10s} {len(per_classe[c]):6d} immagini")

### Com'è fatta un'immagine?


In [ ]:
def apri_immagine(nome_file):
    return Image.open(archivio.open(nome_file)).convert("RGB")

img = apri_immagine(per_classe[CLASSI[0]][0])
arr = np.array(img)

print("Dimensione (larghezza, altezza):", img.size)
print("Come array NumPy -> shape:", arr.shape, "| dtype:", arr.dtype, "| min:", arr.min(), "| max:", arr.max())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(arr); axes[0].set_title("Immagine RGB")
for i, (canale, cmap) in enumerate([("Rosso", "Reds"), ("Verde", "Greens"), ("Blu", "Blues")]):
    axes[i + 1].imshow(arr[:, :, i], cmap=cmap)
    axes[i + 1].set_title(f"Canale {canale}")
for ax in axes:
    ax.axis("off")
plt.show()

### Le 5 varietà a confronto

In [ ]:
fig, axes = plt.subplots(len(CLASSI), 6, figsize=(12, 10))
for r, classe in enumerate(CLASSI):
    for c in range(6):
        axes[r, c].imshow(apri_immagine(per_classe[classe][c]))
        axes[r, c].axis("off")
    axes[r, 0].set_title(classe, loc="left", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Preparare i dati

### 3.1 Caricare un sottoinsieme e ridimensionare

Riduciamo le immagini da 250×250 a **64×64**: 15 volte meno pixel, addestramento molto più veloce, e il chicco si riconosce ancora bene.

In [ ]:
N_PER_CLASSE = 2000     # immagini per classe (in totale 5 × 2000 = 10.000)
IMG_SIZE = 64

def carica_come_array(nome_file):
    return np.array(apri_immagine(nome_file).resize((IMG_SIZE, IMG_SIZE)))

t0 = time.time()
X_list, y_list, file_usati = [], [], set()
for etichetta, classe in enumerate(CLASSI):
    scelti = random.sample(per_classe[classe], N_PER_CLASSE)
    for nome in scelti:
        X_list.append(carica_come_array(nome))
        y_list.append(etichetta)            # la classe diventa un numero: Arborio=0, Basmati=1, ...
    file_usati.update(scelti)

X = np.stack(X_list)       # (N, 64, 64, 3) uint8
y = np.array(y_list)       # (N,)
print(f"Caricate {len(X)} immagini in {time.time() - t0:.0f} s")
print("X:", X.shape, X.dtype, "| y:", y.shape)
print("Etichette:", {i: c for i, c in enumerate(CLASSI)})

### 3.2 Da array a tensore: "channels first"

PyTorch vuole le immagini nel formato **(N, C, H, W)**: numero di immagini, canali, altezza, larghezza. NumPy e PIL usano invece **(N, H, W, C)**. Usiamo `permute` per spostare i canali.

Convertiamo anche i pixel da interi 0–255 a decimali 0–1.

In [ ]:
X_t = torch.tensor(X).permute(0, 3, 1, 2).float() / 255.0     # (N, 64, 64, 3) -> (N, 3, 64, 64)
y_t = torch.tensor(y, dtype=torch.long)                        # le classi sono interi (long)

print("X_t:", X_t.shape, X_t.dtype, "| min", X_t.min().item(), "| max", X_t.max().item())
print("y_t:", y_t.shape, y_t.dtype)

### 3.3 Train, validation e test

Stessa logica del notebook 1: **70% train, 15% validation, 15% test**.

In [ ]:
n = len(X_t)
indici = torch.randperm(n)
n_train, n_val = int(0.70 * n), int(0.15 * n)

train_idx = indici[:n_train]
val_idx = indici[n_train:n_train + n_val]
test_idx = indici[n_train + n_val:]

X_train, y_train = X_t[train_idx], y_t[train_idx]
X_val, y_val = X_t[val_idx], y_t[val_idx]
X_test, y_test = X_t[test_idx], y_t[test_idx]

print(f"Train {len(X_train)} | Validation {len(X_val)} | Test {len(X_test)}")
print("Immagini per classe nel train:", torch.bincount(y_train).tolist())

### 3.4 Normalizzare per canale

Come nel churn: media 0 e deviazione standard 1, **calcolate solo sul train**. Con le immagini si normalizza **per canale** (una media per R, una per G, una per B).

In [ ]:
mean = X_train.mean(dim=(0, 2, 3), keepdim=True)    # media su immagini, altezza e larghezza -> shape (1, 3, 1, 1)
std = X_train.std(dim=(0, 2, 3), keepdim=True)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print("Media per canale:", mean.flatten().round(decimals=3).tolist())
print("Std per canale:  ", std.flatten().round(decimals=3).tolist())

### 3.5 Mini-batch e `DataLoader`

Nel churn usavamo **tutti** i dati a ogni passo (full-batch). Con le immagini non conviene:
- **Memoria:** il dataset completo (75.000 immagini 250×250) occupa più di 50 GB in float32: non sta sulla GPU.
- **Velocità:** con mini-batch da 64 immagini facciamo **centinaia di aggiornamenti per epoca** invece di uno solo.
- **Rumore utile:** il gradiente calcolato su un piccolo batch è un po' "rumoroso", e questo aiuta a non fermarsi in minimi poco buoni.
Il `DataLoader` divide i dati in mini-batch e, con `shuffle=True`, li **rimescola a ogni epoca**.

| Termine | Significato |
|---|---|
| **Batch** | Gruppo di immagini usate per un singolo aggiornamento dei pesi |
| **Iterazione** | Un aggiornamento dei pesi (un batch) |
| **Epoca** | Una passata completa sul train = `N_train / batch_size` iterazioni |

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

xb, yb = next(iter(train_loader))     # prendiamo un batch per vedere com'è fatto
print("Un batch -> immagini:", xb.shape, "| etichette:", yb.shape)
print("Iterazioni per epoca:", len(train_loader))

## 4. Cos'è una convoluzione?

### Perché non usare l'MLP del notebook 1?

Potremmo "srotolare" l'immagine in un vettore di 12.288 numeri e darla a un MLP. Ma:
1. **Troppi parametri:** un solo layer `Linear(12288, 256)` ha più di **3 milioni** di pesi.
2. **Perde la struttura:** srotolando, il modello non sa più quali pixel sono **vicini**.
3. **Non riconosce lo stesso oggetto in posizioni diverse:** un chicco spostato di 10 pixel diventerebbe un input completamente diverso.
### L'idea della convoluzione

Un **filtro** (o *kernel*) è una piccola griglia di pesi, ad esempio 3×3. Lo facciamo **scorrere su tutta l'immagine**: in ogni posizione moltiplichiamo i 9 pixel sotto il filtro per i 9 pesi e sommiamo. Il risultato è una nuova immagine chiamata **feature map**.

In [ ]:
img_prova = X_t[0].mean(dim=0)          # un'immagine in scala di grigi: (64, 64)

filtri = {
    "Bordi verticali": torch.tensor([[1., 0., -1.],
                                     [2., 0., -2.],
                                     [1., 0., -1.]]),
    "Bordi orizzontali": torch.tensor([[1., 2., 1.],
                                       [0., 0., 0.],
                                       [-1., -2., -1.]]),
    "Sfocatura": torch.ones(3, 3) / 9,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(img_prova, cmap="gray"); axes[0].set_title("Originale")
for ax, (nome, k) in zip(axes[1:], filtri.items()):
    # conv2d vuole: input (N, C, H, W) e filtro (C_out, C_in, kH, kW)
    out = F.conv2d(img_prova[None, None], k[None, None], padding=1)
    ax.imshow(out[0, 0], cmap="gray"); ax.set_title(nome)
for ax in axes:
    ax.axis("off")
plt.show()

> **Il punto chiave:** questi filtri li abbiamo scritti noi. In una CNN i valori dei filtri sono **pesi che la rete impara** con la discesa del gradiente, esattamente come i pesi del modello del churn.

### Padding, stride e dimensione dell'output

- **Padding:** aggiunge un bordo di zeri attorno all'immagine, così l'output resta della stessa dimensione (con filtro 3×3 si usa `padding=1`).
- **Stride:** di quanti pixel si sposta il filtro a ogni passo (di solito 1).
$$\text{dimensione output} = \frac{\text{input} - \text{kernel} + 2\cdot\text{padding}}{\text{stride}} + 1$$

Esempio: input 64, kernel 3, padding 1, stride 1 → (64 − 3 + 2) / 1 + 1 = **64**.

### Pooling: rimpicciolire mantenendo l'essenziale

Il **max pooling 2×2** divide l'immagine in quadratini 2×2 e tiene solo il valore **massimo** di ciascuno → l'immagine si **dimezza** in altezza e larghezza.
- Meno numeri da elaborare nei layer successivi.
- Un po' di tolleranza a piccoli spostamenti.

In [ ]:
bordi = F.conv2d(img_prova[None, None], filtri["Bordi verticali"][None, None], padding=1)
ridotta = F.max_pool2d(bordi, kernel_size=2)
ridotta2 = F.max_pool2d(ridotta, kernel_size=2)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, t, titolo in zip(axes, [bordi, ridotta, ridotta2], ["Dopo la convoluzione", "Max pool ×1", "Max pool ×2"]):
    ax.imshow(t[0, 0], cmap="gray")
    ax.set_title(f"{titolo}\n{tuple(t.shape[-2:])}")
    ax.axis("off")
plt.show()

## 5. Costruire la CNN

Struttura tipica: un **estrattore di feature** (blocchi Conv → ReLU → MaxPool) seguito da un **classificatore** (layer `Linear`, come nell'MLP).

```
Input (3, 64, 64)
  → Conv 3→16  + ReLU + MaxPool  → (16, 32, 32)   bordi, colori
  → Conv 16→32 + ReLU + MaxPool  → (32, 16, 16)   forme semplici
  → Conv 32→64 + ReLU + MaxPool  → (64, 8, 8)     forme complesse
  → Flatten                      → 4096
  → Dropout + Linear 4096→128 + ReLU
  → Linear 128→5                 → un punteggio (logit) per varietà
```

Il numero di canali **aumenta** e la dimensione spaziale **diminuisce**: la rete passa da "dove sono i pixel" a "quali caratteristiche ci sono".

In [ ]:
class RiceCNN(nn.Module):
    def __init__(self, n_classi=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64 -> 32
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                  # (64, 8, 8) -> 4096
            nn.Dropout(0.3),               # spegne a caso il 30% dei valori durante il training
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classi),      # 5 logits
        )

    def forward(self, x):
        return self.classifier(self.features(x))

modello = RiceCNN(len(CLASSI)).to(device)    # .to(device): sposta i pesi sulla GPU
print(modello)

### Seguiamo le shape layer per layer

Il modo migliore per capire (e per trovare errori) è far passare un batch finto e stampare la shape dopo ogni layer.

In [ ]:
x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
print(f"{'Input':45s} {tuple(x.shape)}")
for layer in list(modello.features) + list(modello.classifier):
    x = layer(x)
    print(f"{str(layer)[:45]:45s} {tuple(x.shape)}")

### Quanti parametri?

In [ ]:
def conta_parametri(m):
    return sum(p.numel() for p in m.parameters())

mlp_equivalente = nn.Sequential(nn.Flatten(), nn.Linear(3 * IMG_SIZE * IMG_SIZE, 128), nn.ReLU(), nn.Linear(128, 5))

print(f"CNN:                         {conta_parametri(modello):>10,}")
print(f"  di cui nei layer convolutivi: {conta_parametri(modello.features):>7,}")
print(f"MLP con un layer nascosto:   {conta_parametri(mlp_equivalente):>10,}")

> I layer convolutivi, che fanno il lavoro "visivo", hanno pochissimi parametri: è l'effetto della **condivisione dei pesi**. La maggior parte dei parametri è nel primo `Linear`.

---

## 6. Loss per più classi: softmax e cross-entropy

Con 2 classi usavamo la sigmoide (una probabilità). Con **5 classi** il modello produce 5 numeri (logits) e la **softmax** li trasforma in 5 probabilità che **sommano a 1**:

$$p_k = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

La **cross-entropy** guarda solo la probabilità data alla classe **giusta**: $L = -\log(p_{\text{classe giusta}})$. Se è vicina a 1 la loss è quasi 0; se è vicina a 0 la loss è grande.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0, 0.1, 0.3]])     # punteggi inventati per un'immagine
prob = torch.softmax(logits, dim=1)

for c, p in zip(CLASSI, prob[0]):
    print(f"{c:10s} {p:.1%}")
print("Somma:", prob.sum().item())

loss_fn = nn.CrossEntropyLoss()        # include già la softmax: le diamo i logits
for giusta in [0, 2]:
    l = loss_fn(logits, torch.tensor([giusta]))
    print(f"Se la classe giusta è {CLASSI[giusta]:9s} -> loss = {l.item():.3f}")

> Come `BCEWithLogitsLoss`, anche `nn.CrossEntropyLoss` **applica la softmax al suo interno**: il modello restituisce i logits, e la softmax la usiamo solo quando vogliamo leggere le probabilità.

---

## 7. Addestramento con mini-batch

Il ciclo è **lo stesso del notebook 1** (forward → loss → backward → step), con due differenze:
1. C'è un **secondo ciclo** sui mini-batch dentro ogni epoca.
2. Ogni batch va spostato sul **device** (GPU) con `.to(device)`.

In [ ]:
def train_una_epoca(modello, loader, loss_fn, optimizer):
    modello.train()                         # attiva il Dropout
    loss_tot, corretti, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = modello(xb)                # 1. forward
        loss = loss_fn(logits, yb)          # 2. loss
        optimizer.zero_grad()
        loss.backward()                     # 3. backward
        optimizer.step()                    # 4. aggiornamento

        loss_tot += loss.item() * len(yb)
        corretti += (logits.argmax(dim=1) == yb).sum().item()   # argmax = classe con il punteggio più alto
        n += len(yb)
    return loss_tot / n, corretti / n


@torch.no_grad()                            # niente gradienti in valutazione
def valuta(modello, loader, loss_fn):
    modello.eval()                          # disattiva il Dropout
    loss_tot, corretti, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = modello(xb)
        loss_tot += loss_fn(logits, yb).item() * len(yb)
        corretti += (logits.argmax(dim=1) == yb).sum().item()
        n += len(yb)
    return loss_tot / n, corretti / n

> **`model.train()` e `model.eval()`:** alcuni layer si comportano diversamente in training e in uso. Il Dropout spegne neuroni **solo in training**; in valutazione deve usare tutta la rete.

Ora il ciclo completo, con **early stopping** come nel notebook 1.

In [ ]:
def addestra(modello, epochs=20, lr=0.01, momentum=0.9, patience=4, loader=None):
    loader = loader or train_loader
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(modello.parameters(), lr=lr, momentum=momentum)

    storia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    miglior_loss, miglior_pesi, miglior_epoca, attesa = float("inf"), None, 0, 0

    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_una_epoca(modello, loader, loss_fn, optimizer)
        va_loss, va_acc = valuta(modello, val_loader, loss_fn)

        for k, v in zip(storia, [tr_loss, va_loss, tr_acc, va_acc]):
            storia[k].append(v)
        print(f"Epoca {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.1%} | "
              f"val loss {va_loss:.4f} acc {va_acc:.1%} | {time.time() - t0:.1f} s")

        if va_loss < miglior_loss:
            miglior_loss, miglior_epoca, attesa = va_loss, epoch, 0
            miglior_pesi = copy.deepcopy(modello.state_dict())
        else:
            attesa += 1
            if attesa >= patience:
                print(f"Early stopping: nessun miglioramento da {patience} epoche")
                break

    modello.load_state_dict(miglior_pesi)
    storia["best_epoch"] = miglior_epoca
    print(f"Migliore epoca: {miglior_epoca} | val loss {miglior_loss:.4f}")
    return modello, storia


def plot_storia(storia, titolo=""):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, chiave, nome in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuratezza")]:
        ax.plot(storia[f"train_{chiave}"], "o-", label="train")
        ax.plot(storia[f"val_{chiave}"], "o-", label="validation")
        ax.axvline(storia["best_epoch"], color="green", linestyle="--", label="migliore")
        ax.set_title(f"{nome} {titolo}"); ax.set_xlabel("Epoca"); ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
torch.manual_seed(SEED)
modello = RiceCNN(len(CLASSI)).to(device)
modello, storia = addestra(modello, epochs=20, lr=0.01)
plot_storia(storia, "(RiceCNN)")

> 💬 **Da notare:** dopo la **prima epoca** l'accuratezza è già alta. Con il mini-batch, in un'epoca abbiamo fatto più di 100 aggiornamenti dei pesi; nel churn con il full-batch un'epoca era un solo aggiornamento.

---

## 8. Valutazione sul test set

Il dataset è **bilanciato** (stesso numero di immagini per classe), quindi qui l'accuratezza è una metrica affidabile. Guardiamo comunque la matrice di confusione per capire **quali varietà si confondono**.

In [ ]:
@torch.no_grad()
def predici(modello, X):
    modello.eval()
    logits = torch.cat([modello(X[i:i + 256].to(device)).cpu() for i in range(0, len(X), 256)])
    return torch.softmax(logits, dim=1)

prob_test = predici(modello, X_test)
pred_test = prob_test.argmax(dim=1)

print(f"Accuratezza sul test: {(pred_test == y_test).float().mean():.2%}\n")
print(classification_report(y_test, pred_test, target_names=CLASSI, digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=CLASSI).plot(cmap="Blues")
plt.title("Matrice di confusione (test set)")
plt.grid(False)
plt.show()

### Dove sbaglia?

Guardare gli errori è uno dei passi più utili: spesso si scoprono immagini ambigue, etichette sbagliate o limiti del modello.

In [ ]:
errori = torch.where(pred_test != y_test)[0]
print(f"Immagini sbagliate: {len(errori)} su {len(y_test)}")

def mostra(img_norm, ax):
    """Riporta un'immagine normalizzata a valori visualizzabili."""
    img = (img_norm * std[0] + mean[0]).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(img); ax.axis("off")

if len(errori) > 0:
    k = min(8, len(errori))
    fig, axes = plt.subplots(1, k, figsize=(2.2 * k, 2.8), squeeze=False)
    for ax, i in zip(axes[0], errori[:k]):
        mostra(X_test[i], ax)
        ax.set_title(f"Vero: {CLASSI[y_test[i]]}\nPred: {CLASSI[pred_test[i]]} ({prob_test[i].max():.0%})", fontsize=9)
    plt.tight_layout()
    plt.show()

## 9. Cosa "vede" la rete?

Nella sezione 4 abbiamo scritto i filtri a mano. Vediamo cosa producono i filtri **imparati** dalla rete: facciamo passare un'immagine e guardiamo le **feature map** dopo il primo e dopo il terzo blocco convolutivo.


In [ ]:
img = X_test[0:1].to(device)

with torch.no_grad():
    dopo_blocco1 = modello.features[:3](img)[0].cpu()   # Conv+ReLU+Pool -> (16, 32, 32)
    dopo_blocco3 = modello.features(img)[0].cpu()       # tutti e 3 i blocchi -> (64, 8, 8)

fig, axes = plt.subplots(2, 9, figsize=(15, 4))
mostra(X_test[0], axes[0, 0]); axes[0, 0].set_title(CLASSI[y_test[0]])
for i in range(8):
    axes[0, i + 1].imshow(dopo_blocco1[i], cmap="viridis")
    axes[1, i + 1].imshow(dopo_blocco3[i], cmap="viridis")
for ax in axes.flat:
    ax.axis("off")
axes[0, 1].set_title("Blocco 1 (32×32): bordi, contorni, colori", loc="left")
axes[1, 1].set_title("Blocco 3 (8×8): caratteristiche astratte", loc="left")
plt.tight_layout()
plt.show()

> Nel **primo blocco** le feature map assomigliano ancora al chicco (bordi, contorno, zone chiare e scure). Nel **terzo blocco** sono piccole e astratte: non sono più "immagini" per noi, ma per il classificatore contengono l'informazione su forma e dimensione.

---

## 10. Salvare e caricare il modello

Come nel notebook 1: salviamo i pesi **e tutto quello che serve per usarli** (classi, dimensione delle immagini, media e std).

In [ ]:
checkpoint = {
    "model_state": modello.state_dict(),
    "classi": CLASSI,
    "img_size": IMG_SIZE,
    "mean": mean,
    "std": std,
}
torch.save(checkpoint, "rice_cnn.pt")
print(f"Salvato rice_cnn.pt ({os.path.getsize('rice_cnn.pt') / 1e6:.1f} MB)")




In [ ]:

ckpt = torch.load("rice_cnn.pt", map_location=device)

modello_caricato = RiceCNN(len(ckpt["classi"])).to(device)
modello_caricato.load_state_dict(ckpt["model_state"])
modello_caricato.eval()

print("Predizioni identiche:", torch.equal(predici(modello_caricato, X_test).argmax(1), pred_test))

## 11. Inferenza: riconoscere chicchi mai visti

Il dataset ha 75.000 immagini e ne abbiamo usate 10.000. Le altre il modello **non le ha mai viste**: sono perfette per simulare foto nuove.

Una foto nuova arriva come immagine **grezza** (250×250, valori 0–255). Dobbiamo applicare **esattamente** la stessa preparazione:
1. RGB → ridimensionare a 64×64
2. Tensore (C, H, W) con valori 0–1
3. Normalizzare con media e std **del train**
4. Modello → softmax → probabilità

In [ ]:
def prevedi_riso(img_pil, ckpt=ckpt, modello=modello_caricato):
    img = img_pil.convert("RGB").resize((ckpt["img_size"], ckpt["img_size"]))
    x = torch.tensor(np.array(img)).permute(2, 0, 1).float() / 255.0
    x = (x[None] - ckpt["mean"].cpu()) / ckpt["std"].cpu()     # [None]: aggiunge la dimensione del batch
    with torch.no_grad():
        prob = torch.softmax(modello(x.to(device)), dim=1)[0].cpu()
    return dict(zip(ckpt["classi"], prob.tolist()))


def mostra_previsione(img_pil, titolo=""):
    prob = prevedi_riso(img_pil)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1, 2]})
    axes[0].imshow(img_pil); axes[0].axis("off"); axes[0].set_title(titolo)
    colori = ["tab:green" if p == max(prob.values()) else "tab:gray" for p in prob.values()]
    axes[1].barh(list(prob), list(prob.values()), color=colori)
    axes[1].set_xlim(0, 1); axes[1].set_xlabel("Probabilità")
    plt.tight_layout()
    plt.show()

In [ ]:
for classe in CLASSI:
    mai_viste = [f for f in per_classe[classe] if f not in file_usati]
    nome = random.choice(mai_viste)
    mostra_previsione(apri_immagine(nome), f"Vera: {classe}")

### Il modello è robusto?

Nella realtà le foto non saranno mai perfette come quelle del dataset. Proviamo a **modificare** un'immagine: ruotarla, scurirla, ingrandirla.


In [ ]:
nome = random.choice([f for f in per_classe[CLASSI[1]] if f not in file_usati])
originale = apri_immagine(nome)

varianti = {
    "Originale": originale,
    "Ruotata 90°": originale.rotate(90),
    "Più scura": ImageEnhance.Brightness(originale).enhance(0.4),
    "Zoom (crop centrale)": originale.crop((60, 60, 190, 190)).resize((250, 250)),
}
for titolo, img in varianti.items():
    mostra_previsione(img, titolo)

> 💬 **Discussione:** quali modifiche "confondono" il modello? Una rete impara solo da ciò che ha visto nel training. Se nelle foto vere la luce o lo zoom sono diversi, bisogna **includere quelle variazioni nei dati di training** (*data augmentation*, vedi esercizio).

---

## 12. Esercizio

Partite dal template, cambiatelo e confrontate i risultati **sul validation set**.

| Idea | Come |
|---|---|
| Più o meno filtri | `nn.Conv2d(3, 32, ...)`, `nn.Conv2d(32, 64, ...)`… (attenzione: i canali in uscita di un layer = canali in entrata del successivo) |
| Più blocchi convolutivi | aggiungete un blocco Conv → ReLU → MaxPool (e aggiornate la dimensione nel `Linear`!) |
| **Batch normalization** | `nn.BatchNorm2d(n_canali)` dopo ogni `Conv2d`: normalizza le attivazioni dentro la rete, l'addestramento diventa più stabile e veloce |
| Ottimizzatore **Adam** | in `addestra` sostituite `SGD` con `torch.optim.Adam(modello.parameters(), lr=0.001)`: adatta il learning rate di ogni peso automaticamente |
| Immagini più piccole | `IMG_SIZE = 32` (rieseguite dalla sezione 3): quanto si perde? quanto si guadagna in velocità? |
| **Pochi dati** | `N_PER_CLASSE = 50`: compare l'overfitting? |
| **Data augmentation** | nel training usate `augmenta(xb)` (vedi il box qui sotto): ogni batch viene capovolto o ruotato a caso, così la rete vede sempre immagini "nuove" |

> ⚠️ **Errore tipico:** `mat1 and mat2 shapes cannot be multiplied`. Significa che la dimensione nel `nn.Linear` dopo `Flatten` non corrisponde. Usate la cella "Seguiamo le shape" della sezione 5 per trovare il numero giusto.

<details>
<summary>💡 <b>Data augmentation</b> (clicca per aprire)</summary>
L'idea: creare **variazioni realistiche** delle immagini di training, a ogni epoca diverse. La rete non vede mai due volte la stessa immagine identica, quindi è più difficile che la memorizzi.

In [ ]:
def augmenta(xb):
    if torch.rand(1) < 0.5:
        xb = torch.flip(xb, dims=[3])        # specchio orizzontale
    if torch.rand(1) < 0.5:
        xb = torch.flip(xb, dims=[2])        # specchio verticale
    if torch.rand(1) < 0.5:
        xb = torch.rot90(xb, 1, dims=[2, 3]) # rotazione di 90°
    return xb

Per usarla, in `train_una_epoca` sostituite `logits = modello(xb)` con `logits = modello(augmenta(xb))`. **Solo nel training**, mai in validazione o test.
</details>

In [ ]:
# ===================== ESERCIZIO =====================
class MiaCNN(nn.Module):
    def __init__(self, n_classi=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            # TODO: aggiungete blocchi, BatchNorm2d, più filtri...
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 32 * 32, n_classi),   # TODO: aggiornate se cambiate i blocchi sopra
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
mio_modello = MiaCNN(len(CLASSI)).to(device)
print("Parametri:", f"{conta_parametri(mio_modello):,}")

mio_modello, mia_storia = addestra(mio_modello, epochs=20, lr=0.01)    # TODO: provate altri lr
plot_storia(mia_storia, "(la mia CNN)")

In [ ]:
# Confronto sul VALIDATION set
loss_fn = nn.CrossEntropyLoss()
for nome, m in {"RiceCNN": modello, "La mia CNN": mio_modello}.items():
    l, acc = valuta(m, val_loader, loss_fn)
    print(f"{nome:12s} | val loss {l:.4f} | val acc {acc:.2%} | parametri {conta_parametri(m):,}")